In [1]:
# Imports & Configuration
from pathlib import Path

import duckdb
import polars as pl
from great_tables import GT

# Change only this path
SOURCE = Path("../../raw_data/raw_orders_sk.csv").resolve().as_posix()

# DuckDB connection
con = duckdb.connect()

In [2]:
# ============================================================
# Functions
# ============================================================

def show_sql(query: str, max_rows: int = 100):
    """Execute DuckDB SQL and display results using Great Tables."""
    result = con.sql(query).limit(max_rows)

    df = pl.DataFrame(
        result.fetchall(),
        schema=result.columns,
        orient="row"
    )

    display(GT(df))

EXPECTED_CURRENCY = {
    "cz": "CZK",
    "sk": "EUR",
    "hu": "HUF"
}[Path(SOURCE).stem.split("_")[-1]]

In [3]:
# 01 | Source Overview

show_sql(f"""
    SUMMARIZE
    SELECT *
    FROM read_csv('{SOURCE}', all_varchar = true)
""")

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
order_id,VARCHAR,SK-200001,SK-203356,3370,None,None,None,None,None,5439,0.00
line_id,VARCHAR,14530000000000,14530000102630,5204,None,None,None,None,None,5439,0.00
created_at,VARCHAR,2025-12-31T23:00:34Z,2026-09-30T23:58:30Z,3433,None,None,None,None,None,5439,0.00
sku,VARCHAR,FB-BOARD-L,FB-WOK-30-BLK,31,None,None,None,None,None,5439,0.00
quantity,VARCHAR,1,2,2,None,None,None,None,None,5439,0.00
price_with_vat,VARCHAR,10.99,99.99,27,None,None,None,None,None,5439,0.00
discount_with_vat,VARCHAR,0.0,9.9,114,None,None,None,None,None,5439,0.00
currency,VARCHAR,EUR,EUR,1,None,None,None,None,None,5439,0.00
customer_id,VARCHAR,sk-000001,sk-002197,2147,None,None,None,None,None,5439,6.89
cancelled_at,VARCHAR,2026-01-01T04:48:31Z,2026-10-02T14:33:35Z,102,None,None,None,None,None,5439,97.21


In [4]:
# 02 | Data Quality Checks

show_sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_rows,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(order_id), '') IS NULL
               OR NULLIF(TRIM(line_id), '') IS NULL
               OR NULLIF(TRIM(sku), '') IS NULL
               OR NULLIF(TRIM(created_at), '') IS NULL
               OR NULLIF(TRIM(exported_at), '') IS NULL
        ) AS missing_required,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(customer_id), '') IS NULL
        ) AS missing_customer,

        COUNT(*) FILTER (
            WHERE cancelled_at IS NOT NULL
        ) AS cancelled_rows,

        COUNT(*) FILTER (
            WHERE TRY_CAST(quantity AS INTEGER) IS NULL
               OR TRY_CAST(quantity AS INTEGER) <= 0
               OR TRY_CAST(price_with_vat AS DECIMAL(18,2)) IS NULL
               OR TRY_CAST(price_with_vat AS DECIMAL(18,2)) < 0
               OR TRY_CAST(discount_with_vat AS DECIMAL(18,2)) IS NULL
               OR TRY_CAST(discount_with_vat AS DECIMAL(18,2)) < 0
        ) AS invalid_numeric,

        COUNT(*) FILTER (
            WHERE currency IS DISTINCT FROM '{EXPECTED_CURRENCY}'
        ) AS invalid_currency,

        COUNT(*) FILTER (
            WHERE TRY_CAST(created_at AS TIMESTAMPTZ) IS NULL
               OR TRY_CAST(exported_at AS TIMESTAMP) IS NULL
               OR (
                   NULLIF(TRIM(cancelled_at), '') IS NOT NULL
                   AND TRY_CAST(cancelled_at AS TIMESTAMPTZ) IS NULL
               )
        ) AS invalid_timestamps

    FROM read_csv('{SOURCE}', all_varchar = true)
""")

total_rows,duplicate_rows,missing_required,missing_customer,cancelled_rows,invalid_numeric,invalid_currency,invalid_timestamps
5439,263,0,375,152,0,0,0


In [5]:
# 03 | Duplicate Investigation

show_sql(f"""
    WITH versions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY line_id
                ORDER BY exported_at DESC
            ) AS rn
        FROM read_csv(
            '{SOURCE}',
            all_varchar = true
        )
    )
    SELECT
        COUNT(*) AS duplicated_lines,
        COUNT(*) FILTER (
            WHERE old.cancelled_at IS DISTINCT FROM new.cancelled_at
        ) AS changed_cancellations,
        COUNT(*) FILTER (
            WHERE old.quantity IS DISTINCT FROM new.quantity
               OR old.price_with_vat IS DISTINCT FROM new.price_with_vat
               OR old.discount_with_vat IS DISTINCT FROM new.discount_with_vat
        ) AS changed_financial_values
    FROM versions old
    JOIN versions new
        ON old.line_id = new.line_id
       AND old.rn = 2
       AND new.rn = 1
""")

duplicated_lines,changed_cancellations,changed_financial_values
263,118,0


In [6]:
# 04 | Business Validity

show_sql(f"""
    SELECT
        COUNT(*) FILTER (
            WHERE TRIM(order_id) = ''
               OR TRIM(line_id) = ''
               OR TRIM(sku) = ''
               OR TRIM(currency) = ''
        ) AS empty_required,

        COUNT(*) FILTER (
            WHERE TRY_CAST(discount_with_vat AS DECIMAL(18,2))
                > TRY_CAST(price_with_vat AS DECIMAL(18,2))
                * TRY_CAST(quantity AS INTEGER)
        ) AS excessive_discount,

        COUNT(*) FILTER (
            WHERE cancelled_at IS NOT NULL
              AND TRY_CAST(cancelled_at AS TIMESTAMPTZ)
                < TRY_CAST(created_at AS TIMESTAMPTZ)
        ) AS invalid_cancellation,

        COUNT(DISTINCT order_id) AS unique_orders,
        COUNT(DISTINCT sku) AS unique_skus

    FROM read_csv('{SOURCE}', all_varchar = true)
""")

empty_required,excessive_discount,invalid_cancellation,unique_orders,unique_skus
0,0,0,3356,32


In [7]:
# 05 | Schema

show_sql(f"""

    DESCRIBE
    SELECT *
    FROM read_csv_auto('{SOURCE}')
""")

column_name,column_type,null,key,default,extra
order_id,VARCHAR,YES,None,None,None
line_id,BIGINT,YES,None,None,None
created_at,TIMESTAMP WITH TIME ZONE,YES,None,None,None
sku,VARCHAR,YES,None,None,None
quantity,BIGINT,YES,None,None,None
price_with_vat,DOUBLE,YES,None,None,None
discount_with_vat,DOUBLE,YES,None,None,None
currency,VARCHAR,YES,None,None,None
customer_id,VARCHAR,YES,None,None,None
cancelled_at,TIMESTAMP WITH TIME ZONE,YES,None,None,None


In [8]:
# 06 | Sample Data

show_sql(f"""
    SELECT *
    FROM read_csv('{SOURCE}', all_varchar = true)
    LIMIT 5
""")

order_id,line_id,created_at,sku,quantity,price_with_vat,discount_with_vat,currency,customer_id,cancelled_at,exported_at
SK-200026,14530000000772,2025-12-31T23:29:18Z,FB-PAN-28,2,73.99,22.2,EUR,sk-000017,None,2026-01-01 02:00:20
SK-200026,14530000000782,2025-12-31T23:29:18Z,FB-POT-24,1,93.99,14.1,EUR,sk-000017,None,2026-01-01 02:00:20
SK-200018,14530000000526,2025-12-31T23:49:21Z,FB-SET-PAN3,1,157.99,23.7,EUR,sk-000013,None,2026-01-01 02:00:26
SK-200018,14530000000562,2025-12-31T23:49:21Z,FB-SAUCE-16,2,52.99,15.9,EUR,sk-000013,None,2026-01-01 02:00:26
SK-200018,14530000000594,2025-12-31T23:49:21Z,FB-PAN-24,1,60.99,9.15,EUR,sk-000013,None,2026-01-01 02:00:26


| Area | Finding | Recommendation |
|---|---|---|
| Duplicates | 263 duplicate rows, including 118 cancellation updates | Retain the latest export per `line_id` |
| Missing values | 375 missing customer IDs | Preserve NULL values |
| Cancellations | 152 cancelled rows in raw data | Handle in downstream calculations |
| Data validity | No invalid numeric values, timestamps or business values detected | Apply appropriate data types |
| Currency | EUR | Preserve source currency until conversion |

**Source overview:** 5,439 rows, 3,356 unique orders and 32 unique SKUs.

**Expected rows after deduplication:** 5,176.

**Pending checks:** SKU integrity against the product catalog and FX rate coverage.